In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Este es el Notebook de trabajo para post-inferencia

- En este notebook se trabaja para hacer estable el proyecto realizado anteriormente.
- Después de haber hecho el lock del baseline (`LR_smote_0.5`), la tarea que comienza es la capa determinista de post-inferencia con reglas de negocio.
- Aquí se define la arquitectura de gobernanza del producto para el futuro con reglas mínimas en modo `shadow` (es decir, marcan candidatos bloqueados pero no cambian la decisión final del modelo).
- En esta iteración dejamos trazabilidad operativa (`blocked_by_rule_candidate`, `block_reason_candidate`) para ampliar después con decisiones de negocio más robustas.
- Resultado actual: esta capa añade control y auditoría, pero no mejora todavía el Top-1/Top-2 en holdout temporal.
- Dejamos también constancia aquí del contrato y la explicación de la validación que hacemos contra el input diario, al final del notebook.

## Supuestos

- El baseline está bloqueado y se evalúa con su corte temporal oficial (`cutoff_date = 2028-02-21`), sin mover el contrato del champion.
- La métrica principal para decisión de negocio es por evento (`Top-1/Top-2 hit`), no solo por fila.
- El grano operativo se mantiene en 1 fila por `event_id + proveedor_candidato`.
- La capa determinista actual está en modo `shadow`: marca bloqueo pero no cambia score/ranking del modelo.

## Límites

- Las reglas mínimas actuales cubren 10 combinaciones (`PRODUCT_002/PRODUCT_003`, `TERMINAL_001`, `SUPPLIER_020/SUPPLIER_004/SUPPLIER_003/SUPPLIER_035/SUPPLIER_058`), y siguen siendo una política inicial (no global).
- El motor actual solo añade flags (`blocked_by_rule_candidate`, `block_reason_candidate`); aún no decide `decision_final` ni `override_reason` en producción diaria.
- Esta iteración aporta gobernanza y trazabilidad, pero no garantiza mejora de KPI; para mejorar Top-1/Top-2 hay que diseñar reglas dirigidas a errores reales del top del ranking.
- La evaluación “assist simulado” es análisis offline; no equivale todavía a flujo productivo completo con `rules_engine.py` integrado.


## Librerías

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import pandas as pd
import numpy as np
import sys

# Sube de notebooks/ a la raíz del repo

sys.path.append(str(Path.cwd().parent))

from src.ml.shared import functions as fc
from src.ml.product import recommend_supplier as rs

## Carga de datos

In [3]:
# Carga de dataset y checks

import pandas as pd

ROOT = PROJECT_ROOT
DATASET_PATH = ROOT / "data" / "public" / "dataset_modelo_proveedor_v2_candidates.csv"

df = pd.read_csv(DATASET_PATH, dtype=str, keep_default_na=False)
df["target_elegido"] = pd.to_numeric(df["target_elegido"], errors="coerce").fillna(0).astype(int)
df["fecha_evento"] = pd.to_datetime(df["fecha_evento"], errors="coerce")

print("rows:", len(df))
print("events:", df["event_id"].nunique())
print("dup(event_id, proveedor_candidato):", df.duplicated(["event_id", "proveedor_candidato"]).sum())

event_pos = df.groupby("event_id")["target_elegido"].sum()
print("events con 0 positivos:", int((event_pos == 0).sum()))
print("events con >1 positivos:", int((event_pos > 1).sum()))

rows: 155946
events: 15404
dup(event_id, proveedor_candidato): 0
events con 0 positivos: 0
events con >1 positivos: 0


## Análisis y toma de decisiones

In [4]:
# Tabla resumen a nivel de combinación: producto + terminal + proveedor.

tabla_resumen = fc.tabla_resumen(df)
tabla_resumen


,producto_canonico,terminal_compra,proveedor_candidato,rows,events,wins,win_rate_event
377,PRODUCT_003,TERMINAL_001,SUPPLIER_009,7154,7154,4850,0.677942
410,PRODUCT_003,TERMINAL_001,SUPPLIER_050,7141,7141,1640,0.229660
387,PRODUCT_003,TERMINAL_001,SUPPLIER_020,6651,6651,0,0.000000
386,PRODUCT_003,TERMINAL_001,SUPPLIER_019,5959,5959,283,0.047491
413,PRODUCT_003,TERMINAL_001,SUPPLIER_054,5714,5714,8,0.001400
169,PRODUCT_002,TERMINAL_001,SUPPLIER_009,4106,4106,2569,0.625670
209,PRODUCT_002,TERMINAL_001,SUPPLIER_050,4103,4103,1045,0.254692
213,PRODUCT_002,TERMINAL_001,SUPPLIER_054,3935,3935,4,0.001017
179,PRODUCT_002,TERMINAL_001,SUPPLIER_020,3851,3851,0,0.000000
415,PRODUCT_003,TERMINAL_001,SUPPLIER_058,3476,3476,0,0.000000


### Detalles

- Hay varios candidatos en la tabla de resumen para ser incluidos en las reglas mínimas: SUPPLIER_020, SUPPLIER_003, SUPPLIER_035, SUPPLIER_058, SUPPLIER_004 aparecen en el top20 y no fueron elegidas históricamente ninguna vez.

In [5]:
candidatas = [
    ("PRODUCT_002", "TERMINAL_001", "SUPPLIER_020"),
    ("PRODUCT_003", "TERMINAL_001", "SUPPLIER_020"),
    ("PRODUCT_002", "TERMINAL_001", "SUPPLIER_004"),
    ("PRODUCT_003", "TERMINAL_001", "SUPPLIER_004"),
    ("PRODUCT_002", "TERMINAL_001", "SUPPLIER_003"),
    ("PRODUCT_003", "TERMINAL_001", "SUPPLIER_003"),
    ("PRODUCT_002", "TERMINAL_001", "SUPPLIER_035"),
    ("PRODUCT_003", "TERMINAL_001", "SUPPLIER_035"),
    ("PRODUCT_002", "TERMINAL_001", "SUPPLIER_058"),
    ("PRODUCT_003", "TERMINAL_001", "SUPPLIER_058")
]

checks = pd.DataFrame([fc.check_rule_candidate(tabla_resumen, p, t, v, min_events=100) for p, t, v in candidatas])
checks


,producto,terminal,proveedor,events,wins,win_rate_event,pass_soporte,pass_zero_wins,pre_aprobada_shadow
0,PRODUCT_002,TERMINAL_001,SUPPLIER_020,3851,0,0.0,True,True,True
1,PRODUCT_003,TERMINAL_001,SUPPLIER_020,6651,0,0.0,True,True,True
2,PRODUCT_002,TERMINAL_001,SUPPLIER_004,0,0,NaN,False,False,False
3,PRODUCT_003,TERMINAL_001,SUPPLIER_004,2262,0,0.0,True,True,True
4,PRODUCT_002,TERMINAL_001,SUPPLIER_003,1926,0,0.0,True,True,True
5,PRODUCT_003,TERMINAL_001,SUPPLIER_003,3074,0,0.0,True,True,True
6,PRODUCT_002,TERMINAL_001,SUPPLIER_035,1950,0,0.0,True,True,True
7,PRODUCT_003,TERMINAL_001,SUPPLIER_035,3175,0,0.0,True,True,True
8,PRODUCT_002,TERMINAL_001,SUPPLIER_058,2116,0,0.0,True,True,True
9,PRODUCT_003,TERMINAL_001,SUPPLIER_058,3476,0,0.0,True,True,True


## Cierre Day 01 · Decisión de reglas

### Análisis de la decisión

- Se evaluaron candidatas por combinación `producto_canonico + terminal_compra + proveedor_candidato` con criterios:
  - `wins == 0` (nunca elegida históricamente)
  - `events >= 100` (soporte mínimo)
- Las combinaciones aprobadas para `shadow` fueron 10:
  - `SUPPLIER_020`, `SUPPLIER_004`, `SUPPLIER_003`, `SUPPLIER_035`, `SUPPLIER_058` en `PRODUCT_002/PRODUCT_003` para `TERMINAL_001`.
- En holdout temporal del champion, la capa determinista añade trazabilidad (bloqueos y motivos) pero no cambia el KPI principal Top-1/Top-2 en esta iteración.
- Decisión de ingeniería: mantener estas reglas en modo `shadow` como baseline de gobernanza y observabilidad para siguientes iteraciones.

### Resultado esperado del guardado

- Se genera/actualiza automáticamente `config/business_blocklist_rules.csv` con las reglas aprobadas.
- Se guarda además una copia trazable en `artifacts/public/postinferencia/` con timestamp.


In [6]:
# Guardado automático del CSV de reglas aprobado (shadow v1)

fc.guardar_reglas_de_negocio(checks)["rules_export"].head(12)

,rule_id,active,fecha_inicio,fecha_fin,producto_canonico,terminal_canonico,proveedor_canonico,motivo
0,R001,1,2020-01-01,2034-12-31,PRODUCT_002,TERMINAL_001,SUPPLIER_003,shadow_min_v1_historico_0_wins_eventos_ge_1926
1,R002,1,2020-01-01,2034-12-31,PRODUCT_002,TERMINAL_001,SUPPLIER_020,shadow_min_v1_historico_0_wins_eventos_ge_3851
2,R003,1,2020-01-01,2034-12-31,PRODUCT_002,TERMINAL_001,SUPPLIER_035,shadow_min_v1_historico_0_wins_eventos_ge_1950
3,R004,1,2020-01-01,2034-12-31,PRODUCT_002,TERMINAL_001,SUPPLIER_058,shadow_min_v1_historico_0_wins_eventos_ge_2116
4,R005,1,2020-01-01,2034-12-31,PRODUCT_003,TERMINAL_001,SUPPLIER_003,shadow_min_v1_historico_0_wins_eventos_ge_3074
5,R006,1,2020-01-01,2034-12-31,PRODUCT_003,TERMINAL_001,SUPPLIER_004,shadow_min_v1_historico_0_wins_eventos_ge_2262
6,R007,1,2020-01-01,2034-12-31,PRODUCT_003,TERMINAL_001,SUPPLIER_020,shadow_min_v1_historico_0_wins_eventos_ge_6651
7,R008,1,2020-01-01,2034-12-31,PRODUCT_003,TERMINAL_001,SUPPLIER_035,shadow_min_v1_historico_0_wins_eventos_ge_3175
8,R009,1,2020-01-01,2034-12-31,PRODUCT_003,TERMINAL_001,SUPPLIER_058,shadow_min_v1_historico_0_wins_eventos_ge_3476
9,R000,0,2020-01-01,2034-12-31,*,*,DUMMY,Plantilla de ejemplo desactivada


## Integración operativa y decisión de arquitectura

- Se implementa `src/ml/rules/engine.py` para ejecutar post-inferencia en modo `shadow` por CLI y por integración en UI.
- Se extrae la lógica de reglas a `src/ml/rules/blocklist.py` para desacoplar ML de ETL privado y evitar dependencia de `src/etl/*` en ejecución operativa/publicación.
- La app (`app.py`) ahora aplica automáticamente `rules_engine` tras inferencia y exporta CSV con trazabilidad de bloqueos.
- Validación funcional: smoke test interactivo Streamlit end-to-end realizado el `2030-03-04` (usuario), sin incidencias.

## Contrato mínimo input diario (v1)

### 1) Objetivo
Validar que el CSV diario tiene estructura y calidad mínima para inferencia reproducible.

### 2) Grano obligatorio
- 1 fila por (`event_id`, `proveedor_candidato`).
- No se permiten duplicados en esa clave.

### 3) Columnas obligatorias
- `event_id`
- `fecha_evento`
- `proveedor_candidato`
- `producto_canonico`
- `terminal_compra`
- `coste_min_dia_proveedor`
- `rank_coste_dia_producto`
- `terminales_cubiertos`
- `observaciones_oferta`
- `candidatos_evento_count`
- `coste_min_evento`
- `coste_max_evento`
- `spread_coste_evento`
- `delta_vs_min_evento`
- `ratio_vs_min_evento`
- `litros_evento`
- `dia_semana`
- `mes`
- `fin_mes`

### 4) Tipos esperados
- Fecha parseable (`fecha_evento`): formato ISO `YYYY-MM-DD` (aceptar datetime si truncable a fecha).
- Numéricas (float/int):
  - `coste_min_dia_proveedor`, `coste_min_evento`, `coste_max_evento`,
  - `spread_coste_evento`, `delta_vs_min_evento`, `ratio_vs_min_evento`,
  - `litros_evento`, `rank_coste_dia_producto`, `terminales_cubiertos`,
  - `observaciones_oferta`, `candidatos_evento_count`, `dia_semana`, `mes`, `fin_mes`.
- Categóricas string no vacías:
  - `event_id`, `proveedor_candidato`, `producto_canonico`, `terminal_compra`.

### 5) Nulos críticos (FAIL)
- No se permiten nulos en columnas obligatorias.
- Si falta una columna obligatoria -> FAIL directo.

### 6) Reglas de dominio mínimas
- `mes` en `[1..12]`
- `dia_semana` en `[0..6]` (o `[1..7]`, pero elegir una convención y fijarla)
- `fin_mes` en `{0,1}`
- `litros_evento > 0`
- `coste_min_dia_proveedor > 0`
- `rank_coste_dia_producto >= 1`
- `candidatos_evento_count >= 1`
- Coherencia de costes:
  - `coste_min_evento <= coste_max_evento`
  - `spread_coste_evento >= 0`

### 7) Resultado de validación
- `PASS`: sin errores.
- `PASS_WITH_WARNINGS`: errores 0, pero hay avisos no críticos (por ejemplo columnas extra no esperadas).
- `FAIL`: cualquier error crítico.

### 8) Artefactos de salida (obligatorios)
- Reporte JSON por corrida:
  - `artifacts/public/validations/input_daily/<YYYYMMDD>/input_validation_report_<run_id>.json`
- Campos mínimos del reporte:
  - `status`, `rows_total`, `errors`, `warnings`, `error_count`, `warning_count`, `ts_utc`, `input_path`.

### 9) Regla operativa
- Si `FAIL`: no correr inferencia.
- Si `PASS` o `PASS_WITH_WARNINGS`: permitir inferencia y guardar reporte.

## Validación del contrato mínimo y artefactos

1. **Carpeta por día + JSON por ejecución**
- Se crea carpeta diaria en `artifacts/public/validations/input_daily/YYYYMMDD`.
- Dentro se genera un JSON con timestamp UTC en el nombre: `input_validation_report_YYYYMMDDTHHMMSSZ.json`.
- O sea: **fecha en carpeta + fecha/hora en fichero** para trazabilidad fina por corrida.

2. **`inference_input_contract.yaml` = contrato**
- Define qué se exige al input diario: columnas obligatorias, grano, reglas de dominio, coherencia, etc. -> Las reglas definidas en la celda de markdown anterior.

3. **`validate_inference_input.py` = validador**
- Carga el YAML.
- Valida el input (CSV o DataFrame) contra ese contrato.
- Produce el **artefacto de evidencia** (`report.json`) con `PASS / PASS_WITH_WARNINGS / FAIL`, errores, warnings y metadatos.
- Este script está ya enganchado a `app.py`.

Resumen corto:
- **YAML** = reglas.
- **Script** = motor de validación.
- **JSON** = evidencia auditable de cada ejecución.